# Stage 3B.03 — seed-999 end-to-end smoke

In [ ]:
import csv,os,subprocess
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; P=Path.home()/"LIBERO-plus"; ID=Path.home()/"venv-stage1-id/bin/python"; OOD=Path.home()/"venv-stage1-ood/bin/python"; N=Path.home()/"stage1-native"; OUT=Path.home()/"stage3b_smoke"; OUT.mkdir(exist_ok=True); MAN=OUT/"stage3b_smoke_manifest.csv"; AUD=OUT/"stage3b_smoke_pairing.csv"
bench=subprocess.run(["git","-C",str(R),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip(); plus=subprocess.run(["git","-C",str(P),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip()
subprocess.run([str(ID),"-m","async_vla_benchmark.scripts.make_stage3b_smoke_manifest","--output",str(MAN),"--git-sha",bench,"--lerobot-git-sha","2aba372b4e217cc47db28e0f836859b20d1456c9","--libero-plus-git-sha",plus,"--model-revision","8e174154ef5f6c60a8da12ae99c303d8963138c1"],cwd=R,check=True)
base=os.environ.copy(); base.update({"CUDA_VISIBLE_DEVICES":"1","MUJOCO_EGL_DEVICE_ID":"1","MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg"})
for scene,py in (("id",ID),("ood",OOD)):
 env=base.copy();
 if scene=="ood": env.update({"PYTHONPATH":str(P),"MAGICK_HOME":str(N),"PATH":str(N/"bin")+os.pathsep+env.get("PATH",""),"LD_LIBRARY_PATH":str(N/"lib")+os.pathsep+env.get("LD_LIBRARY_PATH","")})
 subprocess.run([str(py),"-m","async_vla_benchmark.scripts.resolve_stage3_initializations","--config",str(R/"async_vla_benchmark/configs/stage3b.yaml"),"--manifest",str(MAN),"--scene",scene,"--expected-rows","4","--expected-cells-per-key","2","--audit-output",str(AUD)],cwd=R,env=env,check=True)
 subprocess.run([str(py),"-u","-m","async_vla_benchmark.scripts.run_stage3b","--config",str(R/"async_vla_benchmark/configs/stage3b.yaml"),"--manifest",str(MAN),"--output-dir",str(OUT),"--scene",scene,"--resume","--verbose"],cwd=R,env=env,check=True)
rows=list(csv.DictReader(open(OUT/"stage3b_episode_results.csv"))); assert len(rows)==4 and len({r['run_id'] for r in rows})==4 and all(r['status'].startswith('ok') for r in rows)
print("PASS: four seed-999 smoke episodes; no analysis seeds used")